In [139]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io

In [140]:
import mlflow
import mlflow.pytorch

In [141]:
mlflow.set_experiment("MLP_Clasificador_Imagenes")

<Experiment: artifact_location='file:///home/_joacoo/Workspace/2C2026/Redes/Skin_DataSet_Classification/mlruns/788045200843311305', creation_time=1780482876458, experiment_id='788045200843311305', last_update_time=1780482876458, lifecycle_stage='active', name='MLP_Clasificador_Imagenes', tags={}>

In [142]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [143]:
# Albumentations Transformations para reducir overfitting
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [144]:
# Función para loguear una figura matplotlib en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)

In [145]:
# Función para matriz de confusión y clasificación
def log_classification_report(model, loader, writer, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.label_encoder.classes_)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=train_dataset.label_encoder.classes_)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)


In [146]:
# Crear directorio de logs
# log_dir = "runs/mlp_experimento_1"
# writer = SummaryWriter(log_dir=log_dir)


In [147]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.image_paths = []
        self.labels = []

        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(cls)

        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label

In [148]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"

In [149]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Resize(64, 64), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Transformaciones de Validación (SIN Data Augmentation, solo normalización)
val_transform = A.Compose([
    A.Resize(64, 64),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

/home/_joacoo/.conda/envs/{redes_env}/lib/python3.12/site-packages/albumentations/core/validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [150]:
train_dataset = CustomImageDataset(train_dir, transform=train_transform)
val_dataset   = CustomImageDataset(val_dir, transform=val_transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)

In [151]:
# # Clase MLPClassifier con inicialización manual. 

# # Se pueden elegir entre "kaiming", "xavier" o "uniform" al crear la instancia del modelo.

# class MLPClassifier(nn.Module):
#     def __init__(self, input_size=64*64*3, num_classes=10, init_type="kaiming", use_bn=False, use_dropout=False):
#         super().__init__()
        
#         layers = []
#         layers.append(nn.Flatten())
        
#         layers.append(nn.Linear(input_size, 512))
#         if use_bn:
#             layers.append(nn.BatchNorm1d(512))
#         layers.append(nn.ReLU())
#         if use_dropout:
#             layers.append(nn.Dropout(p=0.5))
            
#         layers.append(nn.Linear(512, 256))
#         if use_bn:
#             layers.append(nn.BatchNorm1d(256))
#         layers.append(nn.ReLU())
#         if use_dropout:
#             layers.append(nn.Dropout(p=0.5))
            
#         # Capa de salida
#         layers.append(nn.Linear(256, num_classes))
        
#         self.model = nn.Sequential(*layers)
#         self.init_weights(init_type)

#     def init_weights(self, init_type):
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 if init_type == "kaiming":
#                     nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
#                 elif init_type == "xavier":
#                     nn.init.xavier_uniform_(m.weight)
#                 elif init_type == "uniform":
#                     nn.init.uniform_(m.weight, a=-0.05, b=0.05)
#                 nn.init.zeros_(m.bias)
#             elif isinstance(m, nn.BatchNorm1d):
#                 nn.init.ones_(m.weight)
#                 nn.init.zeros_(m.bias)

#     def forward(self, x):
#         return self.model(x)

class MLPClassifier(nn.Module):
    def __init__(self, input_size=64*64*3, num_classes=10, hidden_layers=[512, 256], init_type="kaiming", use_bn=False, use_dropout=False, dropout_p = 0.5):
        super().__init__()
        
        layers = []
        layers.append(nn.Flatten()) # Aplanamos la entrada
        
        # Guardamos la dimensión actual. Arranca siendo el tamaño de la imagen.
        current_dim = input_size
        
        # Construimos las capas ocultas secuencialmente según la lista que nos pasen
        for h_dim in hidden_layers:
            layers.append(nn.Linear(current_dim, h_dim))
            if use_bn:
                layers.append(nn.BatchNorm1d(h_dim)) #
            layers.append(nn.ReLU()) #
            if use_dropout:
                layers.append(nn.Dropout(p=dropout_p)) # Ahora 'p' es dinámico
            
            # El tamaño de salida de esta capa será el de entrada de la que sigue
            current_dim = h_dim
            
        # Capa de salida: conecta la última capa oculta con el número de clases
        layers.append(nn.Linear(current_dim, num_classes)) #
        
        self.model = nn.Sequential(*layers) #
        self.init_weights(init_type) #

    def init_weights(self, init_type):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init_type == "kaiming":
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu') #
                elif init_type == "xavier":
                    nn.init.xavier_uniform_(m.weight) #
                elif init_type == "uniform":
                    nn.init.uniform_(m.weight, a=-0.05, b=0.05) #
                nn.init.zeros_(m.bias) #
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight) #
                nn.init.zeros_(m.bias) #

    def forward(self, x):
        return self.model(x) 

In [152]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train_dataset.labels))

# Configuración de estrategia
ESTRATEGIA_INIT = "xavier"  # Opciones: "kaiming", "xavier", "uniform"
ACTIVAR_BN = True         # True o False
ACTIVAR_DROPOUT = True    # True o False
APLICAR_L2 = True         # True o False

# Directorio de TensorBoard y MLflow
exp_name = f"init_{ESTRATEGIA_INIT}_BN_{ACTIVAR_BN}_Drop_{ACTIVAR_DROPOUT}_L2_{APLICAR_L2}"
log_dir = f"runs/{exp_name}"
writer = SummaryWriter(log_dir=log_dir)

# Cambio el modelo instanciado pasándole la estrategia
model = MLPClassifier(
    num_classes=num_classes, 
    init_type=ESTRATEGIA_INIT, 
    use_bn=ACTIVAR_BN, 
    use_dropout=ACTIVAR_DROPOUT
).to(device)


criterion = nn.CrossEntropyLoss()

# Aplicar Weight Decay (L2) de forma condicional
wd_value = 1e-4 if APLICAR_L2 else 0.0
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=wd_value)

In [153]:
# Entrenamiento y validación
def evaluate(model, loader, epoch=None, prefix="val"):
    log_classification_report(model, val_loader, writer, step=epoch, prefix="val")
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [154]:
# Loop de entrenamiento
n_epochs = 200
with mlflow.start_run():
    # Log hiperparámetros
    mlflow.log_params({
        # --- Arquitectura del Modelo ---
        "model": "MLPClassifier",
        "input_size": 64,
        "use_batch_norm": ACTIVAR_BN,      
        "use_dropout": ACTIVAR_DROPOUT,    
        "dropout_rate": 0.1,  # ¡Importante registrar el valor real!
        
        # --- Configuración del Entrenamiento ---
        "optimizer": "Adam",
        "loss_fn": "CrossEntropyLoss",
        "batch_size": batch_size,
        "lr": 1e-3,  # O 0.0001 según tu segunda lista
        "epochs": n_epochs,
        "weight_decay": wd_value,          
        "es_patience": 5,     # Paciencia del Early Stopping
        
        # --- Data Augmentation (Aumento de Datos) ---
        "h_flip": 0.0, 
        "v_flip": 0.5, 
        "rb_contrast": 0.0, 
        
        # --- Rutas de Datos ---
        "train_dir": train_dir,
        "val_dir": val_dir,
})
    
    # ---------------
    # EARLY STOPPING
    # ---------------
    patience = 20          # Cuántas épocas aguantar sin mejoras
    patience_counter = 0  # Contador de épocas malas consecutivas
    best_val_loss = float('inf') # La mejor pérdida arranca en el infinito
    
    
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0
        
        # Visualizar pesos en la primera época en TensorBoard
        if epoch == 0:
            for name, param in model.named_parameters():
                writer.add_histogram(f"Weights_Init/{name}", param, epoch)
    
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}"):
            images, labels = images.to(device), labels.to(device)
    
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total
        val_loss, val_acc = evaluate(model, val_loader, epoch=epoch, prefix="val")
    
        print(f"Epoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")
    
        writer.add_scalar("train/loss", train_loss, epoch)
        writer.add_scalar("train/accuracy", train_acc, epoch)
    
        # Log en MLflow
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        }, step=epoch)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0  # Resetea el contador porque el modelo mejoró
            # Guarda el mejor estado matemático
            torch.save(model.state_dict(), "best_mlp_model.pth")
            print("  => ¡Nueva mejor pérdida de validación! Modelo guardado.")
        else:
            patience_counter += 1  # No mejoró, consumimos un intento
            print(f"  => La validación no mejoró. Paciencia: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print("¡Early Stopping activado! Deteniendo el entrenamiento de forma segura.")
                break # Sale del bucle 'for epoch' inmediatamente
        

    # Guardar modelo
    print("Cargando el mejor modelo guardado por Early Stopping para el registro final...")
    model.load_state_dict(torch.load("best_mlp_model.pth"))
    
    # Guarda el modelo final en MLflow (modelo óptimo)
    torch.save(model.state_dict(), "best_mlp_model.pth") 
    mlflow.log_artifact("best_mlp_model.pth")
    mlflow.pytorch.log_model(model, artifact_path="pytorch_model")

Epoch 1/200: 100%|██████████| 11/11 [00:03<00:00,  2.76it/s]
/home/_joacoo/.conda/envs/{redes_env}/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/_joacoo/.conda/envs/{redes_env}/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/_joacoo/.conda/envs/{redes_env}/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` paramete

Epoch 1:
  Train Loss: 2.4162, Accuracy: 20.00%
  Val   Loss: 1.9338, Accuracy: 36.09%
  => ¡Nueva mejor pérdida de validación! Modelo guardado.


Epoch 2/200: 100%|██████████| 11/11 [00:03<00:00,  2.81it/s]


Epoch 2:
  Train Loss: 2.0875, Accuracy: 30.81%
  Val   Loss: 1.7096, Accuracy: 39.05%
  => ¡Nueva mejor pérdida de validación! Modelo guardado.


Epoch 3/200: 100%|██████████| 11/11 [00:03<00:00,  2.76it/s]


...

Epoch 78/200: 100%|██████████| 11/11 [00:03<00:00,  2.83it/s]


Epoch 78:
  Train Loss: 1.0055, Accuracy: 60.74%
  Val   Loss: 1.1874, Accuracy: 53.85%
  => La validación no mejoró. Paciencia: 19/20


Epoch 79/200: 100%|██████████| 11/11 [00:03<00:00,  2.86it/s]


Epoch 79:
  Train Loss: 0.9982, Accuracy: 59.41%
  Val   Loss: 1.1568, Accuracy: 56.21%
  => La validación no mejoró. Paciencia: 20/20
¡Early Stopping activado! Deteniendo el entrenamiento de forma segura.
Cargando el mejor modelo guardado por Early Stopping para el registro final...


2026/06/03 09:18:34 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [155]:
!tensorboard --logdir=runs/

# %load_ext tensorboard
# %tensorboard --logdir=runs/

TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.19.0 at http://localhost:6006/ (Press CTRL+C to quit)
^C
